In [ ]:
!pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas
import unsloth

import os
from google.colab import files
uploaded = files.upload()  # upload kaggle.json

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!kaggle competitions download -c dl-spring-2026-svg-generation -p /content/data
!cd /content/data && unzip -o dl-spring-2026-svg-generation.zip
!ls -lh /content/data/*.csv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 159.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 165.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 158.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 145.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

Saving kaggle.json to kaggle.json
100% 32.4M/32.4M [00:00<00:00, 159MB/s] 

Archive:  dl-spring-2026-svg-generation.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
-rw-r--r-- 1 root root 183K Mar 12 04:06 /content/data/sample_submission.csv
-rw-r--r-- 1 root root 152K Mar 12 04:06 /content/data/test.csv
-rw-r--r-- 1 root root 130M Mar 12 04:06 /content/data/train.csv


In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=128,
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✅ Model loaded")
model.print_trainable_parameters()

==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Unsloth 2026.3.17 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ Model loaded
trainable params: 239,468,544 || all params: 3,325,407,232 || trainable%: 7.2012


In [ ]:
import pandas as pd
from datasets import Dataset

train_df = pd.read_csv("/content/data/train.csv")

train_df = train_df[
    train_df['svg'].str.len().between(200, 6000) &
    train_df['svg'].str.startswith('<svg')
].reset_index(drop=True)
print(f"After filtering: {len(train_df)} samples")

SYSTEM_PROMPT = "You are an SVG code generator. Given a description, output valid SVG code only. No explanations."

def format_example(row):
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{row['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{row['svg']}<|im_end|>"
    )

train_df['text'] = train_df.apply(format_example, axis=1)
dataset = Dataset.from_pandas(train_df[['text']])
split = dataset.train_test_split(test_size=0.01, seed=42)
print(f"Train: {len(split['train'])} | Eval: {len(split['test'])}")

After filtering: 47171 samples
Train: 46699 | Eval: 472


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="/content/svg_lora_output",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=200,
    weight_decay=0.01,
    bf16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=500,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    dataset_text_field="text",
    max_seq_length=4096,
    packing=True,
    args=training_args,
)

trainer.train()
print("✅ Training complete!")

Unsloth: Tokenizing ["text"] (num_proc=30):   0%|          | 0/46699 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=30):   0%|          | 0/472 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 46,699 | Num Epochs = 4 | Total steps = 5,840
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 239,468,544 of 3,325,407,232 (7.20% trained)


Step,Training Loss,Validation Loss
200,0.397422,0.387896
400,0.355530,0.355257
600,0.343914,0.342790
800,0.336199,0.335156
1000,0.331396,0.328559
1200,0.315081,0.323144
1400,0.309264,0.317740
1600,0.297845,0.315212
1800,0.308347,0.311892
2000,0.305738,0.309900


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

✅ Training complete!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

ADAPTER_DIR = "/content/drive/MyDrive/svg_3b_adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("✅ Adapter saved")
!ls -lh {ADAPTER_DIR}

Mounted at /content/drive
✅ Adapter saved
total 925M
-rw------- 1 root root 1.2K Mar 28 00:05 adapter_config.json
-rw------- 1 root root 914M Mar 28 00:05 adapter_model.safetensors
-rw------- 1 root root 2.5K Mar 28 00:05 chat_template.jinja
-rw------- 1 root root 5.2K Mar 28 00:05 README.md
-rw------- 1 root root  404 Mar 28 00:05 tokenizer_config.json
-rw------- 1 root root  11M Mar 28 00:05 tokenizer.json
-rw------- 1 root root 5.5K Mar 28 00:05 training_args.bin


In [ ]:
!pip install -q transformers accelerate peft bitsandbytes pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import files
uploaded = files.upload()  # upload kaggle.json

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!kaggle competitions download -c dl-spring-2026-svg-generation -p /content/data
!cd /content/data && unzip -o dl-spring-2026-svg-generation.zip
print("✅ Done")

Mounted at /content/drive


Saving kaggle.json to kaggle.json
100% 32.4M/32.4M [00:04<00:00, 7.77MB/s]

Archive:  dl-spring-2026-svg-generation.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
✅ Done


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/svg_3b_adapter"

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("Loading adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
print(f"✅ Model loaded on {next(model.parameters()).device}")

Loading base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loading adapter...
✅ Model loaded on cuda:0


In [ ]:
import torch, re, pandas as pd
import xml.etree.ElementTree as ET

SYSTEM_PROMPT = "You are an SVG code generator. Given a description, output valid SVG code only. No explanations."

test_df = pd.read_csv("/content/data/test.csv")

for i in range(4):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_df.iloc[i]["prompt"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=2000,
            do_sample=False,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"=== Sample {i+1} ===")
    print(f"Prompt: {test_df.iloc[i]['prompt']}")
    print(f"Length: {len(raw)} chars")
    print(f"Complete: {'</svg>' in raw}")
    print(f"FULL RAW:\n{raw}")
    print()

=== Sample 1 ===
Prompt: firewood stack cut logs wood with leaf illustration.
Length: 2898 chars
Complete: False
FULL RAW:
<svg height="128" viewBox="-5 0 36 36" width="128" xmlns="http://www.w3.org/2000/svg"><path d="m9.47 1c-1.11-.01-2 .89-2 2v1h-1l-1.5 1a1 1 0 0 0 -.29.71c0 .16.01.32.03.47s.04.29.06.42zm-1.47 2h-1v1h1z"/><circle cx="10" cy="10" r="2"/><g fill="#ffccbc"><rect rx=".2" y="10" x="10" height="16" width="4"/><ellipse ry="2" rx="3" transform="matrix(.9866016 -0.17364818 .17364818 .9866016 -1.94 1)"/></g><path d="m10 10q-1.11-.01-2 .89v1h-1l-1.5 1a1 1 0 0 0 -.29.71c0 .16.01.32.03.47s.04.29.06.42zm-1.47 2h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><path d="m10 10h-1v1h1z"/><p